# Python并发编程（协程篇）## 学习目标- 理解并发与并行的概念- 理解事件循环模型- 掌握async/await语法- 理解Task任务的概念和使用- 掌握协程中的错误处理和超时取消- 掌握生产者消费者模型的协程实现- 理解协程与多线程的适用场景

## 一、核心概念### 1. 什么是协程？协程是实现**并发编程**的一种方式。与多线程/多进程不同，协程是单线程的，**由用户决定在何处交出控制权**。### 2. 事件循环（Event Loop）事件循环启动一个统一的调度器，让调度器来决定一个时刻去运行哪个任务。相比于多线程，省去了启动线程、管理线程、同步锁等各种开销。### 3. async/await语法糖- `async` 声明一个函数为异步函数- `await` 在协程中等待另一个协程完成- 使用 `asyncio.run()` 触发运行

## 二、从一个爬虫说起### 1. 同步版本的爬虫（低效）

In [ ]:
"""同步版本的爬虫"""import timedef crawl_page(url):print(f"crawling {url}")sleep_time = int(url.split("_")[-1])time.sleep(sleep_time)print(f"OK {url}")def main(urls):for url in urls:crawl_page(url)# 测试（爬取4个页面，分别用时1-4秒）# main(["url_1", "url_2", "url_3", "url_4"])# 总耗时：10秒

### 2. 协程版本的爬虫（await同步调用）

In [ ]:
"""协程版本（await同步调用）"""import asyncioasync def crawl_page(url):print(f"crawling {url}")sleep_time = int(url.split("_")[-1])await asyncio.sleep(sleep_time)print(f"OK {url}")async def main(urls):for url in urls:await crawl_page(url)# asyncio.run(main(["url_1", "url_2", "url_3", "url_4"]))# 总耗时仍然是10秒，因为await是阻塞的！# 相当于用异步接口写了同步代码

## 三、任务（Task）### 1. 使用create_task实现真正的并发**要点**：- `asyncio.create_task()` 创建任务，创建后很快会被调度执行- 任务不会阻塞主协程- 需要使用 `await task` 等待所有任务完成- `asyncio.gather(*tasks)` 可以同时等待多个任务

In [ ]:
"""使用create_task实现并发"""import asyncioasync def crawl_page(url):print(f"crawling {url}")sleep_time = int(url.split("_")[-1])await asyncio.sleep(sleep_time)print(f"OK {url}")async def main(urls):# 创建所有任务tasks = [asyncio.create_task(crawl_page(url)) for url in urls]# 方法1：逐个等待for task in tasks:await task# asyncio.run(main(["url_1", "url_2", "url_3", "url_4"]))# 总耗时：4秒（等于最长的那个爬虫）"""使用gather同时等待"""import asyncioasync def main_with_gather(urls):tasks = [asyncio.create_task(crawl_page(url)) for url in urls]# 方法2：使用gatherawait asyncio.gather(*tasks)# *tasks 解包列表，将列表变成函数的参数# asyncio.run(main_with_gather(["url_1", "url_2", "url_3", "url_4"]))

## 四、解密协程运行时### 1. 同步await的执行流程

In [ ]:
"""方式1：直接await（同步执行）"""import asyncioasync def worker_1():print("worker_1 start")await asyncio.sleep(1)print("worker_1 done")async def worker_2():print("worker_2 start")await asyncio.sleep(2)print("worker_2 done")async def main():print("before await")await worker_1()  # 等待worker_1完成print("awaited worker_1")await worker_2()  # 再等待worker_2完成print("awaited worker_2")# asyncio.run(main())# 总耗时：3s（顺序执行）

### 2. create_task的并发执行流程

In [ ]:
"""方式2：create_task（并发执行）"""import asyncioasync def main():task1 = asyncio.create_task(worker_1())task2 = asyncio.create_task(worker_2())print("before await")await task1print("awaited worker_1")await task2print("awaited worker_2")# asyncio.run(main())# 总耗时：2s（并发执行）## 执行流程分析：# 1. task1和task2被创建，进入事件循环# 2. await task1，切换到worker_1# 3. worker_1执行到await asyncio.sleep(1)，切换到worker_2# 4. worker_2执行到await asyncio.sleep(2)，事件调度器暂停# 5. 1秒后，worker_1完成，回到主任务# 6. await task2，等待worker_2完成# 7. 2秒后，worker_2完成，所有任务结束

## 五、错误处理和超时取消### 1. 使用gather + return_exceptions**关键参数**：`return_exceptions=True`- 设置为True时，异常会作为结果返回，不会影响其他任务- 不设置时，异常会抛出到调用层，取消其他未完成的任务

In [ ]:
"""错误处理和超时取消"""import asyncioasync def worker_1():await asyncio.sleep(1)return 1async def worker_2():await asyncio.sleep(2)return 2 / 0  # 除零错误async def worker_3():await asyncio.sleep(3)return 3async def main():task_1 = asyncio.create_task(worker_1())task_2 = asyncio.create_task(worker_2())task_3 = asyncio.create_task(worker_3())await asyncio.sleep(2)task_3.cancel()  # 超时取消res = await asyncio.gather(task_1, task_2, task_3, return_exceptions=True)print(res)# 输出：[1, ZeroDivisionError(), CancelledError()]# asyncio.run(main())

## 六、生产者消费者模型使用 `asyncio.Queue` 实现协程版的生产者消费者模型。

In [ ]:
"""协程版生产者消费者模型"""import asyncioimport randomasync def consumer(queue, id):"""消费者"""while True:val = await queue.get()print(f"{id} get a val: {val}")await asyncio.sleep(1)async def producer(queue, id):"""生产者"""for i in range(5):val = random.randint(1, 10)await queue.put(val)print(f"{id} put a val: {val}")await asyncio.sleep(1)async def main():queue = asyncio.Queue()# 创建任务consumer_1 = asyncio.create_task(consumer(queue, "consumer_1"))consumer_2 = asyncio.create_task(consumer(queue, "consumer_2"))producer_1 = asyncio.create_task(producer(queue, "producer_1"))producer_2 = asyncio.create_task(producer(queue, "producer_2"))await asyncio.sleep(10)consumer_1.cancel()consumer_2.cancel()await asyncio.gather(consumer_1, consumer_2,producer_1, producer_2,return_exceptions=True)# asyncio.run(main())

## 七、练习与自测### 练习题1：并发下载**题目**：编写一个协程程序，并发下载3个网页并计算总用时。**提示**：使用 `asyncio.gather()` 和 `aiohttp`（或 `asyncio.sleep()` 模拟）。

In [ ]:
"""练习题1解答：模拟并发下载"""import asyncioimport timeasync def download_site(name, delay):"""模拟下载一个网站"""print(f"开始下载 {name}... (需要{delay}秒)")await asyncio.sleep(delay)print(f"{name} 下载完成！")return f"{name}的内容"async def main():sites = [("百度", 2),("谷歌", 3),("GitHub", 1),]start = time.time()tasks = [asyncio.create_task(download_site(name, delay)) for name, delay in sites]results = await asyncio.gather(*tasks)elapsed = time.time() - startprint(f"\n结果：{results}")print(f"总耗时：{elapsed:.2f}秒")  # 约3秒（等于最长的那个）# asyncio.run(main())

### 练习题2：超时控制**题目**：创建3个协程任务，运行时间分别为1秒、5秒、2秒。要求只等待2秒，超过2秒的任务自动取消。**提示**：使用 `task.cancel()` 或 `asyncio.wait_for()`。

In [ ]:
"""练习题2解答：超时控制"""import asyncioasync def my_task(name, delay):print(f"{name} 开始，预计{delay}秒")await asyncio.sleep(delay)print(f"{name} 完成！")return f"{name}的结果"async def main():# 创建3个任务tasks = [asyncio.create_task(my_task("任务A", 1)),asyncio.create_task(my_task("任务B", 5)),  # 超时asyncio.create_task(my_task("任务C", 2)),]await asyncio.sleep(2)  # 等待2秒# 取消未完成的任务for task in tasks:if not task.done():task.cancel()print(f"已取消：{task}")# 收集结果results = await asyncio.gather(*tasks, return_exceptions=True)for i, r in enumerate(results):if isinstance(r, asyncio.CancelledError):print(f"任务被取消")else:print(f"结果：{r}")# asyncio.run(main())

## 八、知识总结### 重点记忆1. **协程的核心**- `async def` 定义协程函数- `await` 等待另一个协程- `asyncio.run()` 触发事件循环- `asyncio.create_task()` 创建任务实现并发2. **三种调用方式**- `await coroutine`：同步执行（阻塞）- `asyncio.create_task(coroutine)`：创建任务（非阻塞）- `asyncio.gather(*tasks)`：同时等待多个任务3. **协程 vs 多线程**- 协程：单线程，用户控制切换- 多线程：多线程，操作系统控制切换（可能竞态条件）- 协程适合I/O密集型，多进程适合CPU密集型4. **错误处理**- `return_exceptions=True` 让异常作为结果返回- `task.cancel()` 取消任务- `asyncio.gather()` 收集所有结果和异常

## 深入Asyncio### 1. Sync（同步）VS Async（异步）| 特性 | Sync | Async ||------|------|------|| 执行方式 | 一个接一个执行 | 交替执行 || 等待行为 | 阻塞等待 | 被block时切换 || 效率 | 低（浪费等待时间） | 高（充分利用时间） |**生活类比**：- Sync：等待报表生成→等5分钟→再写邮件 → 浪费时间- Async：先写邮件→报表生成后切换查看→继续写 → 高效利用时间

In [ ]:
"""Async vs Sync 对比"""# Sync版：一个一个来def sync_work():print("开始任务A")time.sleep(2)  # 等待2秒print("任务A完成")print("开始任务B")time.sleep(1)print("任务B完成")# 总耗时：3秒# Async版：交替执行import asyncioasync def task_a():print("开始任务A")await asyncio.sleep(2)print("任务A完成")async def task_b():print("开始任务B")await asyncio.sleep(1)print("任务B完成")async def async_work():await asyncio.gather(task_a(), task_b())# 总耗时：2秒（max(2,1)）

### 2. Asyncio 事件循环的底层细节Event Loop维护两个任务列表：- **预备状态列表**：任务空闲，随时待命- **等待状态列表**：任务正在等待外部操作（如I/O）**执行流程**：1. event loop从预备状态列表选取一个任务执行2. 任务遇到await → 交还控制权给event loop3. event loop根据任务完成情况放入对应列表4. 遍历等待列表检查是否完成5. 重复循环直到所有任务完成**关键优势**：Asyncio的任务在运行时不会被外部因素打断，不会出现race condition！

In [ ]:
"""查看asyncio.run的底层实现"""# asyncio.run() 是Python 3.7+引入的语法糖# 等价于老版本写法：# loop = asyncio.get_event_loop()# try:#     loop.run_until_complete(coro)# finally:#     loop.close()import asyncioasync def hello():print("Hello")# 新写法（推荐）asyncio.run(hello())# 老写法（Python 3.7之前）# loop = asyncio.get_event_loop()# loop.run_until_complete(hello())# loop.close()

### 3. Asyncio的缺陷1. **库兼容性问题**：很多库不支持Asyncio- requests不兼容 → 需用aiohttp- 需要特定的异步库支持2. **任务调度更复杂**- 需要自己决定用gather还是wait- 需要理解run_until_complete和run_forever的区别### 4. 多线程 vs Asyncio 选择指南```if I/O bound:if I/O slow（大量并发任务）:Use Asyncioelse（有限任务）:Use multi-threadingelif CPU bound:Use multi-processing```